# Лекция 7. Часть 1 - Полиморфизм и инкапсуляция

Практические примеры и задания.

**Как работать:** запускайте ячейки по порядку и смотрите на вывод. В ячейках с заданиями пишите свой код.

## 1. Класс и экземпляр

Класс - описание структуры. Экземпляр - конкретный объект, созданный по классу.

In [ ]:
# Пример 1: минимальный класс с атрибутами
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance


acc = BankAccount("Олег", 7500.0)
print(acc.owner, acc.balance)

In [ ]:
# Пример 2: метод меняет состояние объекта
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount


acc = BankAccount("Олег", 7500.0)
acc.deposit(2500.0)
print(acc.balance)

In [ ]:
# Пример 3: __str__ задаёт читаемый вид объекта
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

    def __str__(self):
        return f"Счёт {self.owner}: {self.balance} сум"


print(BankAccount("Олег", 7500.0))

### Задание 1

Создайте класс `Card` с атрибутами `holder` и `number` и методом `__str__`, возвращающим строку вида `Карта Asel: 8600...`. Создайте объект и выведите его через `print()`.

In [ ]:
# ваш код здесь

## 2. Наследование и super()

Наследник переиспользует код родителя. `super().__init__(...)` вызывает конструктор родителя.

In [ ]:
# Пример 1: наследник получает методы родителя
class Account:
    def __init__(self, owner):
        self.owner = owner

    def info(self):
        print("Владелец:", self.owner)


class SavingsAccount(Account):
    pass


SavingsAccount("Олег").info()   # метод достался от Account

In [ ]:
# Пример 2: super() расширяет конструктор родителя
class BasicCard:
    def __init__(self, holder, number):
        self.holder = holder
        self.number = number


class PremiumCard(BasicCard):
    def __init__(self, holder, number, cashback):
        super().__init__(holder, number)   # поля родителя
        self.cashback = cashback           # своё поле


vip = PremiumCard("Asel", "8600...", 0.05)
print(vip.holder, vip.cashback)

In [ ]:
# Пример 3: наследник добавляет собственный метод
class PremiumCard(BasicCard):
    def __init__(self, holder, number, cashback):
        super().__init__(holder, number)
        self.cashback = cashback

    def bonus(self, amount):
        return amount * self.cashback


vip = PremiumCard("Asel", "8600...", 0.05)
print("Кэшбэк:", vip.bonus(100000))

### Задание 2

На основе `BasicCard` создайте класс `CorporateCard` с дополнительным атрибутом `company`. В `__init__` сначала вызовите `super().__init__(...)`. Создайте объект и выведите `holder` и `company`.

In [ ]:
# ваш код здесь

## 3. Полиморфизм: переопределение методов

Класс-наследник задаёт свою версию метода с тем же именем. Вызывающий код не знает конкретный класс.

In [ ]:
# Пример 1: потомок переопределяет метод родителя
class PaymentGateway:
    def process(self, amount):
        print("Базовая обработка:", amount)


class UzCardGateway(PaymentGateway):
    def process(self, amount):
        print("UzCard списывает:", amount)


UzCardGateway().process(150000)

In [ ]:
# Пример 2: super() - сначала логика родителя, потом своя
class UzCardGateway(PaymentGateway):
    def process(self, amount):
        super().process(amount)         # общая логика
        print("UzCard: генерация QR")   # своя логика


UzCardGateway().process(150000)

In [ ]:
# Пример 3: один цикл - разные классы
class HumoGateway(PaymentGateway):
    def process(self, amount):
        print("Humo списывает:", amount)


for gw in [UzCardGateway(), HumoGateway()]:
    gw.process(50000)                   # вызов один, поведение разное

### Задание 3

Добавьте класс `MillyBankGateway` - наследник `PaymentGateway`. Переопределите `process`: вызовите `super()`, затем выведите `Milly Bank: онлайн-платёж {amount}`. Прогоните объект в цикле вместе с другими шлюзами.

In [ ]:
# ваш код здесь

## 4. Утиная типизация

Наследование не нужно. Достаточно, чтобы у объекта был метод с нужным именем.

In [ ]:
# Пример 1: два несвязанных класса с одним методом
class ApplePay:
    def charge(self, amount):
        print("Apple Pay:", amount)


class GooglePay:
    def charge(self, amount):
        print("Google Pay:", amount)


def pay(service, amount):
    service.charge(amount)              # важно лишь наличие charge()


pay(ApplePay(), 50000)
pay(GooglePay(), 50000)

In [ ]:
# Пример 2: в цикл попадают любые объекты с методом charge()
for wallet in [ApplePay(), GooglePay()]:
    wallet.charge(30000)

In [ ]:
# Пример 3: если метода нет - ошибка во время выполнения
class EmptyService:
    pass


try:
    pay(EmptyService(), 1000)
except AttributeError as e:
    print("Ошибка:", e)

### Задание 4

Создайте класс `QrPayService` с методом `charge(self, amount)`, который выводит `QR: оплата {amount}`. Класс ничего не наследует. Передайте объект в функцию `pay`.

In [ ]:
# ваш код здесь

## 5. MRO - порядок разрешения методов

Когда метод есть у нескольких родителей, Python ищет его по списку `__mro__` слева направо.

In [ ]:
# Пример 1: цепочка наследования и __mro__
class A:
    pass


class B(A):
    pass


print([c.__name__ for c in B.__mro__])

In [ ]:
# Пример 2: ромб - чей метод сработает
class Base:
    def check(self):
        print("Base")


class Left(Base):
    def check(self):
        print("Left")


class Right(Base):
    def check(self):
        print("Right")


class Both(Left, Right):
    pass


Both().check()                          # первый по MRO - Left
print([c.__name__ for c in Both.__mro__])

### Задание 5

Поменяйте порядок родителей: `class Both(Right, Left)`. Что выведет `Both().check()`? Проверьте и объясните результат в комментарии.

In [ ]:
# ваш код здесь

## 6. Инкапсуляция: _ и __

`_имя` - соглашение «внутреннее, не трогать снаружи». `__имя` - Name Mangling: имя меняется на `_Класс__имя`.

In [ ]:
# Пример 1: _имя - защищённый атрибут по соглашению
class Config:
    def __init__(self):
        self._token = "secret-123"      # снаружи трогать не стоит


cfg = Config()
print(cfg._token)                       # технически доступно

In [ ]:
# Пример 2: __имя - Name Mangling скрывает атрибут
class User:
    def __init__(self, pin):
        self.__pin = pin


u = User("7721")
try:
    print(u.__pin)                      # имени __pin не существует
except AttributeError as e:
    print("Ошибка:", e)

In [ ]:
# Пример 3: настоящее имя после mangling
print(u._User__pin)                     # обходной путь
print(u.__dict__)                       # видно реальное имя атрибута

### Задание 6

Создайте класс `Account` с публичным атрибутом `owner` и приватным `__balance` (начальное значение 0). Проверьте через `try / except`, что обращение к `account.__balance` вызывает `AttributeError`.

In [ ]:
# ваш код здесь

## 7. @property и сеттер

`@property` - чтение как атрибут, но через метод (без скобок). Сеттер проверяет значение перед записью.

In [ ]:
# Пример 1: @property - геттер только для чтения
class Contract:
    def __init__(self, number):
        self._number = number

    @property
    def number(self):
        return self._number


c = Contract("UZ-001")
print(c.number)                         # обращение без скобок

In [ ]:
# Пример 2: сеттер проверяет значение перед записью
class CreditLimit:
    def __init__(self, limit):
        self._limit = limit

    @property
    def limit(self):
        return self._limit

    @limit.setter
    def limit(self, value):
        if value < 0:
            raise ValueError("Лимит не может быть отрицательным")
        self._limit = value


c = CreditLimit(500000)
c.limit = 800000
print(c.limit)
try:
    c.limit = -100
except ValueError as e:
    print("Ошибка:", e)

In [ ]:
# Пример 3: вычисляемое свойство
class Loan:
    def __init__(self, principal, rate):
        self.principal = principal
        self.rate = rate

    @property
    def total(self):
        return self.principal * (1 + self.rate)


loan = Loan(1000000, 0.20)
print(loan.total)

### Задание 7

Создайте класс `Temperature`: значение храните в `_celsius`, `@property celsius` для чтения, сеттер `celsius`, который отклоняет значения ниже `-273.15` (`raise ValueError`). Проверьте на корректном и некорректном значении.

In [ ]:
# ваш код здесь

## 8. Атрибут класса и атрибут экземпляра

Атрибут класса - один на всех экземпляров. Атрибут экземпляра (`self.x`) - у каждого объекта свой.

In [ ]:
# Пример 1: атрибут класса общий для всех объектов
class Loan:
    base_rate = 0.20                    # атрибут класса

    def __init__(self, client):
        self.client = client            # атрибут экземпляра


a = Loan("Тимур")
b = Loan("Елена")
print(a.base_rate, b.base_rate)

In [ ]:
# Пример 2: меняем атрибут класса - меняется у всех
Loan.base_rate = 0.22
print(a.base_rate, b.base_rate)         # 0.22 у обоих

In [ ]:
# Пример 3: счётчик объектов на атрибуте класса
class Player:
    count = 0

    def __init__(self, name):
        self.name = name
        Player.count += 1


Player("A")
Player("B")
print("Создано объектов:", Player.count)

### Задание 8

В классе `Loan` добавьте атрибут класса `count = 0` и увеличивайте `Loan.count` в `__init__`. Создайте 3 кредита и выведите `Loan.count` (должно быть 3).

In [ ]:
# ваш код здесь